# SI4006 · Sesión 1 — Demo de capacidades modernas de IA
### Tópicos Especiales y Aplicaciones en IA · Universidad EAFIT · Semana 1

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manularrea/EAFIT-SI4006/blob/main/sesiones/s01/S01_Demo_Capacidades.ipynb)

---

**Qué es esto:** el notebook de *demostración en vivo* del Bloque 4. No explica mecanismos — muestra **qué van a poder construir al final del semestre**. Son tres demos encadenadas:

1. **Q&A con un LLM pequeño** — el modelo responde... y alucina cuando no sabe.
2. **RAG sobre un documento** — le damos contexto y ahora responde bien.
3. **Análisis multimodal** — el modelo *ve* una imagen y razona sobre ella.

> ⚙️ **Ejecutar en Colab con GPU:** `Runtime → Change runtime type → T4 GPU`. Correr las celdas de arriba hacia abajo.

> 🛟 **Plan B (sin internet / sin GPU):** cada demo tiene, al final, una celda markdown con el resultado esperado para narrarlo. La narrativa importa más que la ejecución.

## 0 · Setup

> ⚠️ **Corre SOLO esta celda de instalación, primero y una única vez.** Todas las dependencias van fijadas aquí para que nada re-instale `transformers` a mitad del notebook.
> Si Colab te muestra el botón **“RESTART RUNTIME”**, haz clic, y vuelve a ejecutar desde esta celda hacia abajo. No instales `transformers` en ninguna otra celda.

In [ ]:
# Instalación única y con versiones fijas (Colab). ~2-3 min la primera vez.
# transformers 4.49 soporta Qwen2.5 (texto), SmolVLM (multimodal) y el pipeline 'image-text-to-text'.
%pip -q install "transformers==4.49.0" "accelerate>=0.34" "sentence-transformers>=3.0" sentencepiece pillow num2words

In [ ]:
import torch, textwrap
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'CPU (activa la GPU en Colab)')

def wrap(t, w=90):
    print('\n'.join(textwrap.fill(l, w) for l in str(t).split('\n')))

## 1 · Demo — Q&A con un LLM open-source pequeño

Cargamos **Qwen2.5-3B-Instruct** (pequeño, multilingüe, corre en la GPU gratuita de Colab).
Le hacemos dos preguntas: una general, y una específica de un dominio que **no** conoce bien.

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
llm = pipeline('text-generation', model=MODEL_ID,
               torch_dtype=torch.float16, device_map='auto')

def ask(question, system='Eres un asistente conciso. Responde en español.', max_new=180):
    msgs = [{'role':'system','content':system},{'role':'user','content':question}]
    out = llm(msgs, max_new_tokens=max_new, do_sample=False)
    return out[0]['generated_text'][-1]['content']

In [ ]:
# (a) Pregunta general — el modelo la responde bien
wrap(ask('¿Qué es la fotosíntesis, en dos frases?'))

In [ ]:
# (b) Pregunta específica de dominio — observen cómo alucina con seguridad
#     (cambien la pregunta por algo de SU dominio: una norma, un producto interno, un caso local)
wrap(ask('¿Qué establece exactamente el Artículo 384 del Código Sustantivo del Trabajo colombiano?'))

> 🙋 **Pregunta al grupo:** el modelo no sabe (o se inventa) la respuesta correcta de la segunda pregunta.
> **¿Cómo se la enseñamos sin reentrenarlo desde cero?**

La respuesta es la siguiente demo: **RAG**.

<details><summary>🛟 Plan B — resultado esperado</summary>

(a) Responde correctamente qué es la fotosíntesis.
(b) Da una respuesta *fluida y con formato de ley* pero **incorrecta o inventada** — el clásico 'alucina con seguridad'.
</details>

## 2 · Demo — RAG sobre un documento

Le damos al modelo el **contexto** que le faltaba. Pipeline mínimo:

`documento → chunking → embeddings → vector store → retrieval → prompt aumentado → respuesta`

Para que la demo nunca falle por una descarga, el 'documento' va embebido abajo.
Pueden reemplazarlo por un PDF real con `pypdf` (celda opcional al final).

In [ ]:
# Nuestro 'documento' de dominio (haría de PDF). Reemplazable por un corpus real.
DOCUMENTO = [
    'El Artículo 384 del Código Sustantivo del Trabajo trata sobre el fuero sindical y su duración.',
    'El fuero sindical protege a ciertos trabajadores para que no sean despedidos sin justa causa calificada por el juez.',
    'La jornada máxima legal en Colombia se ha venido reduciendo gradualmente por la Ley 2101 de 2021 hasta 42 horas semanales.',
    'El contrato de trabajo puede ser verbal o escrito, y a término fijo, indefinido, por obra o labor.',
    'Las horas extra diurnas se remuneran con un recargo del 25 por ciento sobre el valor ordinario.',
]
print(len(DOCUMENTO), 'fragmentos cargados')

In [ ]:
# Embeddings + 'vector store' (cosine en numpy — en M3 usaremos Chroma de verdad)
# (sentence-transformers ya se instaló en la celda de setup — no reinstalar aquí)
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('intfloat/multilingual-e5-small', device=DEVICE)
doc_emb = embedder.encode(['passage: '+d for d in DOCUMENTO], normalize_embeddings=True)

def retrieve(query, k=2):
    q = embedder.encode(['query: '+query], normalize_embeddings=True)
    sims = (doc_emb @ q.T).ravel()
    idx = sims.argsort()[::-1][:k]
    return [DOCUMENTO[i] for i in idx]

In [ ]:
# La MISMA pregunta de antes, ahora con contexto recuperado
pregunta = '¿Qué establece el Artículo 384 del Código Sustantivo del Trabajo?'
contexto = retrieve(pregunta)
print('Contexto recuperado:'); [print(' •', c) for c in contexto]

prompt = f'''Usa SOLO el siguiente contexto para responder. Si no está, dilo.

Contexto:
{chr(10).join('- '+c for c in contexto)}

Pregunta: {pregunta}'''

print('\n— Respuesta con RAG —')
wrap(ask(prompt))

> ✅ Ahora responde **bien** — porque le dimos el contexto correcto, sin reentrenar nada.
> *"Esto que ven es lo que van a saber construir en cuatro semanas, en sus dominios. Y mejor que esto."*

<details><summary>🛟 Plan B — resultado esperado</summary>
El modelo ahora responde apoyándose en el fragmento sobre el fuero sindical, en vez de inventar.
</details>

In [ ]:
# Liberar memoria de la GPU antes de la demo multimodal
import gc
del llm
gc.collect(); torch.cuda.empty_cache() if DEVICE=='cuda' else None
print('Memoria liberada')

## 3 · Demo — Análisis multimodal (el modelo *ve*)

Cargamos **SmolVLM-Instruct**, un modelo visión-lenguaje ligero y bien soportado por `transformers` (API estándar, sin `trust_remote_code`). Le pasamos una imagen y le hacemos preguntas.
Usen una imagen relevante a su dominio: un diagrama, una radiografía abierta, una página escaneada.

In [ ]:
# Usamos el pipeline de alto nivel 'image-text-to-text': maneja processor + modelo
# internamente y su nombre de tarea es estable entre versiones de transformers.
from transformers import pipeline
from PIL import Image
import requests, io

vlm = pipeline('image-text-to-text', model='HuggingFaceTB/SmolVLM-Instruct',
               torch_dtype=torch.float16, device_map='auto')

In [ ]:
# Imagen de ejemplo (reemplazar por una de su dominio: subir con Files o URL)
url = 'http://images.cocodataset.org/val2017/000000039769.jpg'  # dos gatos en un sofá (imagen estable de ejemplo)
img = Image.open(io.BytesIO(requests.get(url).content)).convert('RGB')
img.thumbnail((768, 768)); img

In [ ]:
def ver(imagen, pregunta, max_new=120):
    msgs = [{'role': 'user', 'content': [
        {'type': 'image', 'image': imagen},
        {'type': 'text', 'text': pregunta}]}]
    out = vlm(text=msgs, max_new_tokens=max_new)
    return out[0]['generated_text'][-1]['content'].strip()

for q in ['Describe la imagen en una frase.', '¿Cuántos animales hay y qué están haciendo?']:
    print('Q:', q)
    print('A:', ver(img, q), '\n')

> 👁️ El modelo **ve** y **razona** sobre lo que ve.

<details><summary>🛟 Plan B — resultado esperado</summary>
Describe correctamente la escena de la imagen y responde preguntas específicas sobre su contenido.
</details>

## Cierre

> Estas tres demos son la **versión simple** de lo que van a construir.
> En sus proyectos, los tres elementos van a **coexistir en un solo sistema** — evaluado rigurosamente, desplegado y eficiente.

**Tarea para S02:** terminar la plantilla del proyecto · crear cuenta de Hugging Face · verificar Colab con GPU.

---
### Anexo opcional — cargar un PDF real (en vez del documento embebido)

In [ ]:
# %pip -q install pypdf
# from pypdf import PdfReader
# reader = PdfReader('mi_documento.pdf')  # subir con el panel Files de Colab
# texto = '\n'.join(p.extract_text() or '' for p in reader.pages)
# # chunking simple por párrafos:
# DOCUMENTO = [c.strip() for c in texto.split('\n\n') if len(c.strip()) > 40]
# doc_emb = embedder.encode(['passage: '+d for d in DOCUMENTO], normalize_embeddings=True)
# print(len(DOCUMENTO), 'fragmentos')